# 02 · Local TimesFM 2.5 forecast demo

Zero-shot TimesFM 2.5 forecast over the configured horizon, built through
`geaptimes.models.factory.ForecastFactory`, plotted against the held-out **TEST** actuals.

- Reads the prepped table from BigQuery and downloads the TimesFM checkpoint on first use
  (needs ADC auth + network).
- Univariate (covariates are opt-in/deferred). Quantile band uses the standardized
  `QUANTILES = [0.1, 0.3, 0.5, 0.7, 0.9]`.
- Run with the **geapTimes (uv 3.11)** kernel.

### Colab (optional)

To run in Colab instead of the local `uv` kernel, uncomment the cell below to install the
package and authenticate. Keep it commented for the local flow.

In [ ]:
# !pip install -q -e ..
# from google.colab import auth; auth.authenticate_user()

In [ ]:
from geaptimes.models.base import QUANTILE_COLUMNS
from geaptimes.models.factory import ForecastFactory
from geaptimes.naming import table_names
from geaptimes.schemas import ExperimentConfig

cfg = ExperimentConfig.from_yaml("../config/base_config.yaml")
model_cfg = next(m for m in cfg.models if m.enabled and m.params.type == "timesfm")
model_cfg.name, cfg.forecast.horizon

In [ ]:
# Build the forecaster and run a zero-shot forecast (fit() is a no-op for TimesFM).
forecaster = ForecastFactory.create(model_cfg, cfg)
forecaster.fit()
result = forecaster.predict()
print(result.metadata)
result.predictions.head(10)

In [ ]:
# Fetch the held-out TEST actuals from the prepped table for comparison.
from google.cloud import bigquery

prepped = table_names(cfg)["prepped"]
prepped_id = f"{cfg.project.id}.{cfg.project.bq_dataset}.{prepped}"
data = cfg.data
sql = (
    f"SELECT {data.series_column} AS series, {data.time_column} AS date, "
    f"{data.target_column} AS actual "
    f"FROM `{prepped_id}` WHERE splits = 'TEST'"
)
actuals = bigquery.Client(project=cfg.project.id).query(sql).to_dataframe()
actuals.head()

In [ ]:
# Plot forecast vs TEST actuals (with the q10-q90 band) for one station.
import matplotlib.pyplot as plt

station = result.predictions["series"].iloc[0]
pred = result.predictions[result.predictions["series"] == station].sort_values("date")
act = actuals[actuals["series"] == station].sort_values("date")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(act["date"], act["actual"], "o-", label="actual (TEST)", color="black")
ax.plot(pred["date"], pred["forecast"], "s--", label="forecast", color="tab:blue")
ax.fill_between(pred["date"], pred["q10"], pred["q90"], alpha=0.2, color="tab:blue", label="q10-q90")
ax.set_title(f"TimesFM 2.5 — {station}")
ax.set_xlabel("date")
ax.set_ylabel(cfg.data.target_column)
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print("quantile columns:", QUANTILE_COLUMNS)